# Brazilian E-commerce Analysis

This notebook completes the required Brazilian E-commerce mini-project using pandas, SQLite, and SQL.

## Analytical conventions used

Some assignment prompts are underspecified, so this notebook states the interpretation used:

- **Query 1 percentage:** five-star January 2018 orders ÷ all January 2018 orders.
- **Query 2 customer trend:** distinct `customer_unique_id` values purchasing in each year.
- **Query 3 average order value:** average of total item-price value per order.
- **Query 4–5 revenue:** sum of `price` from `olist_order_items`.
- **Query 6 successful sellers:** show goods sold, revenue, distinct customers, and five-star orders; sort by revenue.
- **Query 7 delivery success:** delivered orders ÷ all orders in each state.
- **Query 8 preferred payment:** payment type used by the most distinct orders in each product category.
- **Query 9 distance:** customer-city to seller-city straight-line distance using ZIP-prefix coordinates and the Haversine formula.


## 1. Setup and data loading

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, inspect

# In Google Colab, upload the original ZIP when prompted.
try:
    from google.colab import files
    uploaded = files.upload()
    zip_filename = list(uploaded.keys())[0]
except ImportError:
    # For local/Jupyter execution, place the ZIP in the current working directory.
    zip_filename = "Brazilian E-Commerce Dive Deep using SQL.zip"

extract_path = "/content/ecommerce" if os.path.exists("/content") else "./ecommerce"

with zipfile.ZipFile(zip_filename, "r") as archive:
    archive.extractall(extract_path)

# Find every CSV even if the archive contains nested folders.
csv_files = {}
for folder, subfolders, filenames in os.walk(extract_path):
    for filename in filenames:
        if filename.endswith(".csv"):
            csv_files[filename] = os.path.join(folder, filename)

table_files = {
    "olist_customers": "olist_customers_dataset.csv",
    "olist_sellers": "olist_sellers_dataset.csv",
    "olist_order_reviews": "olist_order_reviews_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_products_dataset": "olist_products_dataset.csv",
    "olist_geolocation": "olist_geolocation_dataset.csv",
    "product_category_name_translation": "product_category_name_translation.csv",
    "olist_orders": "olist_orders_dataset.csv",
    "olist_order_payments": "olist_order_payments_dataset.csv",
}

dataframes = {
    table_name: pd.read_csv(csv_files[filename])
    for table_name, filename in table_files.items()
}

engine = create_engine("sqlite://", echo=False)

for table_name, df in dataframes.items():
    df.to_sql(table_name, con=engine, index=False, if_exists="replace")

print("Loaded tables:")
print(inspect(engine).get_table_names())


In [ ]:
# Quick verification
pd.read_sql_query("SELECT * FROM olist_customers LIMIT 5", con=engine)


## 2. Query 1 — January 2018 orders with review score 5

Interpretation: percentage = five-star January orders ÷ all January 2018 orders.


In [ ]:
query1 = '''
SELECT
    (
        SELECT COUNT(DISTINCT o.order_id)
        FROM olist_orders AS o
        JOIN olist_order_reviews AS r
            ON o.order_id = r.order_id
        WHERE r.review_score = 5
          AND o.order_purchase_timestamp >= '2018-01-01'
          AND o.order_purchase_timestamp < '2018-02-01'
    ) AS five_star_orders,

    (
        SELECT COUNT(DISTINCT o.order_id)
        FROM olist_orders AS o
        WHERE o.order_purchase_timestamp >= '2018-01-01'
          AND o.order_purchase_timestamp < '2018-02-01'
    ) AS january_orders,

    (
        (
            SELECT COUNT(DISTINCT o.order_id)
            FROM olist_orders AS o
            JOIN olist_order_reviews AS r
                ON o.order_id = r.order_id
            WHERE r.review_score = 5
              AND o.order_purchase_timestamp >= '2018-01-01'
              AND o.order_purchase_timestamp < '2018-02-01'
        ) * 100.0
        /
        (
            SELECT COUNT(DISTINCT o.order_id)
            FROM olist_orders AS o
            WHERE o.order_purchase_timestamp >= '2018-01-01'
              AND o.order_purchase_timestamp < '2018-02-01'
        )
    ) AS percentage;
'''

pd.read_sql_query(query1, con=engine)


## 3. Query 2 — Distinct customer purchase trend year-on-year

In [ ]:
query2 = '''
SELECT
    strftime('%Y', o.order_purchase_timestamp) AS purchase_year,
    COUNT(DISTINCT c.customer_unique_id) AS unique_customers
FROM olist_orders AS o
JOIN olist_customers AS c
    ON o.customer_id = c.customer_id
GROUP BY purchase_year
ORDER BY purchase_year;
'''

pd.read_sql_query(query2, con=engine)


## 4. Query 3 — Average order value

First aggregate item prices to one total per order, then average those order totals.


In [ ]:
query3 = '''
WITH order_totals AS (
    SELECT
        order_id,
        SUM(price) AS order_value
    FROM olist_order_items
    GROUP BY order_id
)
SELECT
    AVG(order_value) AS average_order_value
FROM order_totals;
'''

pd.read_sql_query(query3, con=engine)


## 5. Query 4 — Top 5 cities with highest revenue, 2016–2018

In [ ]:
query4 = '''
SELECT
    c.customer_city,
    SUM(oi.price) AS total_revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON oi.order_id = o.order_id
JOIN olist_customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_purchase_timestamp >= '2016-01-01'
  AND o.order_purchase_timestamp < '2019-01-01'
GROUP BY c.customer_city
ORDER BY total_revenue DESC
LIMIT 5;
'''

pd.read_sql_query(query4, con=engine)


## 6. Query 5 — State-wise revenue, 2016–2018

In [ ]:
query5 = '''
SELECT
    c.customer_state,
    SUM(oi.price) AS total_revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON oi.order_id = o.order_id
JOIN olist_customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_purchase_timestamp >= '2016-01-01'
  AND o.order_purchase_timestamp < '2019-01-01'
GROUP BY c.customer_state
ORDER BY total_revenue DESC;
'''

pd.read_sql_query(query5, con=engine)


## 7. Query 6 — Successful sellers

Metrics shown:
- goods sold = number of order-item rows;
- total revenue = sum of item prices;
- customer count = distinct underlying customers;
- five-star orders = distinct seller orders with review score 5.

The assignment does not define one composite success score, so the result is sorted by revenue.


In [ ]:
query6 = '''
WITH seller_sales AS (
    SELECT
        oi.seller_id,
        COUNT(*) AS goods_sold,
        SUM(oi.price) AS total_revenue,
        COUNT(DISTINCT c.customer_unique_id) AS customer_count
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON oi.order_id = o.order_id
    JOIN olist_customers AS c
        ON o.customer_id = c.customer_id
    GROUP BY oi.seller_id
),
seller_ratings AS (
    SELECT
        oi.seller_id,
        COUNT(DISTINCT CASE
            WHEN r.review_score = 5 THEN oi.order_id
        END) AS five_star_orders
    FROM olist_order_items AS oi
    JOIN olist_order_reviews AS r
        ON oi.order_id = r.order_id
    GROUP BY oi.seller_id
)
SELECT
    s.seller_id,
    s.goods_sold,
    s.total_revenue,
    s.customer_count,
    COALESCE(r.five_star_orders, 0) AS five_star_orders
FROM seller_sales AS s
LEFT JOIN seller_ratings AS r
    ON s.seller_id = r.seller_id
ORDER BY s.total_revenue DESC
LIMIT 10;
'''

pd.read_sql_query(query6, con=engine)


## 8. Query 7 — Delivery success rate across states

In [ ]:
query7 = '''
SELECT
    c.customer_state,
    COUNT(*) AS total_orders,
    SUM(CASE
        WHEN o.order_status = 'delivered' THEN 1
        ELSE 0
    END) AS delivered_orders,
    AVG(CASE
        WHEN o.order_status = 'delivered' THEN 1.0
        ELSE 0.0
    END) * 100 AS delivery_success_rate
FROM olist_orders AS o
JOIN olist_customers AS c
    ON c.customer_id = o.customer_id
GROUP BY c.customer_state
ORDER BY delivery_success_rate DESC;
'''

pd.read_sql_query(query7, con=engine)


## 9. Query 8 — Preferred payment form by product category

Interpretation: the preferred payment type is the one used by the greatest number of distinct orders in each product category.


In [ ]:
query8 = '''
WITH payment_counts AS (
    SELECT
        p.product_category_name,
        pay.payment_type,
        COUNT(DISTINCT oi.order_id) AS payment_count
    FROM olist_order_items AS oi
    JOIN olist_order_payments AS pay
        ON oi.order_id = pay.order_id
    JOIN olist_products_dataset AS p
        ON oi.product_id = p.product_id
    WHERE p.product_category_name IS NOT NULL
    GROUP BY
        p.product_category_name,
        pay.payment_type
),
ranked_payments AS (
    SELECT
        product_category_name,
        payment_type,
        payment_count,
        ROW_NUMBER() OVER (
            PARTITION BY product_category_name
            ORDER BY payment_count DESC
        ) AS rn
    FROM payment_counts
)
SELECT
    product_category_name,
    payment_type,
    payment_count
FROM ranked_payments
WHERE rn = 1
ORDER BY product_category_name;
'''

pd.read_sql_query(query8, con=engine)


## 10. Query 9 — Distance between customer and seller cities

Interpretation: calculate straight-line geographic distance from the customer's location to the seller's location.

The SQL portion assembles one representative coordinate pair per ZIP prefix. Python/NumPy then applies the Haversine formula.


In [ ]:
query9 = '''
WITH geo AS (
    SELECT
        geolocation_zip_code_prefix AS zip_code_prefix,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng
    FROM olist_geolocation
    GROUP BY geolocation_zip_code_prefix
)
SELECT
    oi.order_id,
    c.customer_city,
    s.seller_city,
    cg.lat AS customer_lat,
    cg.lng AS customer_lng,
    sg.lat AS seller_lat,
    sg.lng AS seller_lng
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON oi.order_id = o.order_id
JOIN olist_customers AS c
    ON o.customer_id = c.customer_id
JOIN olist_sellers AS s
    ON oi.seller_id = s.seller_id
JOIN geo AS cg
    ON c.customer_zip_code_prefix = cg.zip_code_prefix
JOIN geo AS sg
    ON s.seller_zip_code_prefix = sg.zip_code_prefix;
'''

distance_data = pd.read_sql_query(query9, con=engine)

lat1 = np.radians(distance_data["customer_lat"])
lon1 = np.radians(distance_data["customer_lng"])
lat2 = np.radians(distance_data["seller_lat"])
lon2 = np.radians(distance_data["seller_lng"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))
earth_radius_km = 6371

distance_data["distance_km"] = earth_radius_km * c

distance_data[
    ["order_id", "customer_city", "seller_city", "distance_km"]
].head(10)


In [ ]:
# Optional summary of calculated distances
distance_data["distance_km"].describe()


## 11. Reusable SQL patterns used in this project

- `SELECT / FROM / WHERE` — retrieve and filter rows.
- `JOIN ... ON ...` — connect related tables through shared keys.
- `GROUP BY` + `COUNT / SUM / AVG` — aggregate at a chosen grain.
- `CASE WHEN` — conditional logic inside SQL.
- `WITH ... AS (...)` — create named intermediate results (CTEs).
- `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` — rank rows within groups.
